<a href="https://colab.research.google.com/github/Viggo-Kristensen/kaggle-competitions/blob/main/spaceship-titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. Problem Definition**

### Predict which passengers are transported to a different dimension. Predictions are made from a set of personal records recovered from the spaceship's damaged computer.

## **2. Imports**

In [109]:
import zipfile
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## **3. Data Loading**

### Uploading Files

In [110]:
from google.colab import files
uploaded = files.upload()

Saving spaceship-titanic.zip to spaceship-titanic (2).zip


### Unzipping Files

In [111]:
zip_path = "spaceship-titanic.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall("data")

os.listdir("data/spaceship-titanic")

['train.csv', '.ipynb_checkpoints', 'sample_submission.csv', 'test.csv']

### Creating pandas Dataframes

In [112]:
train = pd.read_csv("data/spaceship-titanic/train.csv")
test = pd.read_csv("data/spaceship-titanic/test.csv")

## **4. Data Info**

In [113]:
train.info()
train.isnull().sum()
train.head(1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False


## **5. Data Splitting**

In [114]:
X = train.drop("Transported", axis=1)
y = train["Transported"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = X_train.copy()
X_val = X_val.copy()

# **6. Cleaning Data/Preprocessing**

### Filling missing values with median and "unknown"

In [115]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "bool"]).columns

num_medians = X_train[num_cols].median()

X_train[num_cols] = X_train[num_cols].fillna(num_medians)
X_val[num_cols] = X_val[num_cols].fillna(num_medians)

X_train[cat_cols] = X_train[cat_cols].fillna("unknown")
X_val[cat_cols] = X_val[cat_cols].fillna("unknown")

## **7. Feature Engineering**

### Creating a total spend column

In [116]:
money_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

X_train["TotalSpend"] = X_train[money_cols].sum(axis=1)
X_val["TotalSpend"] = X_val[money_cols].sum(axis=1)

### Splitting Cabin into deck, num and side

In [117]:
X_train[["Deck", "Num", "Side"]] = X_train["Cabin"].fillna("Unknown/0/Unknown").str.split("/", expand=True)
X_val[["Deck", "Num", "Side"]] = X_val["Cabin"].fillna("Unknown/0/Unknown").str.split("/", expand=True)

X_train = X_train.drop("Cabin", axis=1)
X_val = X_val.drop("Cabin", axis=1)

## **8. Encoding**

### Create dummies for categorical columns

In [118]:
X_train = pd.get_dummies(X_train)
X_val = pd.get_dummies(X_val)

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

## **9. Training**

### Logistic regression

In [119]:
model = LogisticRegression(max_iter=1500)
model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1500)

## **10. Evaluation**

### Logistic regression

In [120]:
val_preds = model.predict(X_val)
accuracy_score(y_val, val_preds)

0.7705577918343876

## **12. Submission File**

### preprocessing test data

In [121]:
test[num_cols] = test[num_cols].fillna(num_medians)
test[cat_cols] = test[cat_cols].fillna("unknown")

test["TotalSpend"] = test[money_cols].sum(axis=1)

test[["Deck", "Num", "Side"]] = test["Cabin"].fillna("Unknown/0/Unknown").str.split("/", expand=True)
test = test.drop("Cabin", axis=1)

test = pd.get_dummies(test)
test = test.reindex(columns=X_train.columns, fill_value=0)



In [124]:
test_preds = model.predict(test)

In [126]:
submission = pd.read_csv("data/spaceship-titanic/sample_submission.csv")
submission["Transported"] = test_preds.astype(bool)

submission.to_csv("submission.csv", index=False)

## **13. Reflections**


**Notes**
*   Logistic Regression with simple data cleaning consisting of replacing missing numbers with the median and NaN with "unknown" served as a good basis model achieving 76,4% on the validation set.
*   Combining the money features into a TotalSpend column and splitting the cabin into Deck/Num/Side lifted the accuracy of the model to 77,6%.
*   The final model got an accuracy of 78,7% placing me number #1886 in the competition

**What i learned**
*   The classic steps that goes into a simple Kaggle project.
*   How pd.dummies() are used to feed categorical data into Logistic Regression.
*   Different types of feature engineering






